# 02 — Feature Engineering & Chuan Hoa
Log transform - Label Encoding - Train/Test Split - StandardScaler - Roi rac hoa

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data.loader      import load_config
from src.features.builder import FeatureBuilder

cfg      = load_config("../configs/params.yaml")
df_clean = pd.read_parquet("../data/processed/yield_clean.parquet")
print(f"Loaded: {df_clean.shape}")

## 2.1 Feature Engineering

In [ ]:
fb = FeatureBuilder(cfg)
fb.fit_transform(df_clean)
print("Features:", fb.features_)
print(f"Train: {len(fb.X_train):,}  Test: {len(fb.X_test):,}")

## 2.2 Log Transform

In [ ]:
print("Skewness pesticides_tonnes:", round(df_clean["pesticides_tonnes"].skew(), 3))
print("Skewness pesticides_log:  ", round(fb.df_fe_["pesticides_log"].skew(), 3))
print("Skewness Yield:           ", round(df_clean["Yield"].skew(), 3))
print("Skewness Yield_log:       ", round(fb.df_fe_["Yield_log"].skew(), 3))

## 2.3 Kiem tra StandardScaler

In [ ]:
check = pd.DataFrame(fb.X_train_s, columns=fb.features_)
print("Sau StandardScaler (mean~0, std~1):")
print(check.describe().loc[["mean","std"]].round(3))

In [ ]:
fig, axes = plt.subplots(2, len(fb.features_), figsize=(20,7))
for i, feat in enumerate(fb.features_):
    axes[0,i].hist(fb.X_train.iloc[:,i], bins=40, color="steelblue",
                   edgecolor="white", alpha=0.8)
    axes[0,i].set_title(f"{feat}\nTRUOC", fontsize=7, fontweight="bold")
    axes[1,i].hist(fb.X_train_s[:,i], bins=40, color="coral",
                   edgecolor="white", alpha=0.8)
    axes[1,i].set_title(f"{feat}\nSAU scale", fontsize=7, fontweight="bold")
axes[0,0].set_ylabel("Truoc Scale", fontweight="bold")
axes[1,0].set_ylabel("Sau Scale",   fontweight="bold")
plt.suptitle("Phan phoi Features Truoc vs Sau StandardScaler", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 2.4 Roi rac hoa

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18,4))
for ax, (col, title, color) in zip(axes, [
    ("Rain_cat","Luong mua","#3498db"),
    ("Temp_cat","Nhiet do","#e74c3c"),
    ("Pest_cat","Thuoc tru sau","#2ecc71"),
    ("Yield_cat","Nang suat","#9b59b6"),
]):
    counts = fb.df_disc_[col].value_counts().sort_index()
    bars = ax.bar(range(len(counts)), counts.values, color=color,
                  alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=35, ha="right", fontsize=8)
    ax.set_title(title, fontweight="bold")
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                str(v), ha="center", fontsize=7)
plt.suptitle("Phan phoi sau roi rac hoa", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fb.df_fe_.to_parquet("../data/processed/yield_features.parquet", index=False)
fb.df_disc_.to_parquet("../data/processed/yield_discrete.parquet", index=False)
print("Saved features & discretized data")